## Setting up Google Drive access AND installing kaggle

In [1]:
# set up Google Drive Access
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!mkdir -p ~/.kaggle

In [3]:
!ls /content/drive/MyDrive/DLF_2025/

kaggle.json


In [4]:
!cp /content/drive/MyDrive/DLF_2025/kaggle.json ~/.kaggle

In [5]:
!chmod 600 ~/.kaggle/kaggle.json

In [6]:
!pip install kaggle

In [7]:
# Test kaggle
!kaggle datasets list

ref                                                           title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------------------  -------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
ahmeduzaki/global-earthquake-tsunami-risk-assessment-dataset  Global Earthquake-Tsunami Risk Assessment Dataset       16151  2025-10-01 16:35:53.273000           7128        278  1.0              
jaderz/hospital-beds-management                               Hospital Beds Management                                47583  2025-10-03 09:21:58.590000           5769        161  1.0              
ahmadrazakashif/bmw-worldwide-sales-records-20102024          BMW Worldwide Sales Records (2010–2024)                853348  2025-09-20 14:39:45.280000          11966        263  1.0              
grandmaster07/s

## Preparing a dataset

We will be preparing a dataset - for creating a prototype of garbage detection system. We are tasked in improving a recycling system in Poland (new machines) by using the magic of AI. The problem - we have one day to produce a viable prototype, otherwise no funding.

Let's simplify the problem then - we need a quick dataset, for trash, that is both recyclable and not. Quick look at Kaggle - nothing small pops up, BUT we can maybe combine two small datasets?

We were able to find this: https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification (original dataset here - https://huggingface.co/datasets/garythung/trashnet/tree/main) and this: https://www.kaggle.com/datasets/techsash/waste-classification-data

One has recyclables (split to multiple classes), and one has recyclables and organic waste. By combining these datasets, and simplifying it all to two classes we can create a prototype for investors fast.



In [5]:
!mkdir -p datasets

In [6]:
! cd datasets && kaggle datasets download -d asdasdasasdas/garbage-classification && kaggle datasets download techsash/waste-classification-data

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'


In [7]:
!cd datasets && unzip -q garbage-classification.zip

unzip:  cannot find or open garbage-classification.zip, garbage-classification.zip.zip or garbage-classification.zip.ZIP.


In [8]:
!cd datasets && unzip -q waste-classification-data.zip


unzip:  cannot find or open waste-classification-data.zip, waste-classification-data.zip.zip or waste-classification-data.zip.ZIP.


In [9]:
!cd datasets && rm -rf dataset && rm -rf "Garbage classification"

In [10]:
 !cd datasets && mv DATASET waste_classification_data

mv: cannot stat 'DATASET': No such file or directory


In [11]:
!cd datasets && mv "garbage classification"/"Garbage classification" garbage_classification

mv: cannot stat 'garbage classification/Garbage classification': No such file or directory


In [12]:
!cd datasets && rm -rf "garbage classification"

In [13]:
!cd datasets && rm one-indexed-files-notrash_test.txt && rm one-indexed-files-notrash_train.txt && rm one-indexed-files-notrash_val.txt && rm -rf one-indexed-files.txt && rm -rf zero-indexed-files.txt

rm: cannot remove 'one-indexed-files-notrash_test.txt': No such file or directory


## Creating Pytroch dataset and dataloaders

We can now write a simple script, to create a proper Dataset class for our pytorch model.
This will be a few simple python loading operations. The dataset is small, so we could load it to memory, but it is better to just read it from disk JIT.


In [14]:
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from PIL import Image
import os
import random


# For reproducibility
random.seed(42)

IMAGE_OUTPUT_SIZE = (224, 224)


class CombinedWasteDataset(Dataset):
    def __init__(self, root_dir="datasets", num_classes=2, split='train', transform=None):
        """
        Combined dataset from garbage_classification and waste_classification_data.

        All data is combined and split globally into train/val/test (70%/15%/15%).

        Classes: 0=organic, 1=recyclable, 2=trash (if num_classes=3)
        class 3 (trash) is not very numerous (few examples compared to others).
        """
        self.root_dir = root_dir
        self.num_classes = num_classes
        self.split = split
        self.transform = transform
        self.data = []
        self._load_data()

    def _load_data(self):
        self.all_data = []
        # Load from garbage_classification
        garbage_dir = os.path.join(self.root_dir, 'garbage_classification')
        for category in os.listdir(garbage_dir):
            cat_path = os.path.join(garbage_dir, category)
            if os.path.isdir(cat_path):
                label = self._get_garbage_classification_label(category)
                if label is not None:
                    for f in os.listdir(cat_path):
                        if f.endswith('.jpg'):
                            img_path = os.path.join(cat_path, f)
                            self.all_data.append((img_path, label))

        # Load from waste_classification_data
        waste_dir = os.path.join(self.root_dir, 'waste_classification_data')
        for phase in ['TRAIN', 'TEST']:
            for category in ['O', 'R']:
                label = 0 if category == 'O' else 1
                cat_path = os.path.join(waste_dir, phase, category)
                for f in os.listdir(cat_path):
                    if f.endswith('.jpg'):
                        img_path = os.path.join(cat_path, f)
                        self.all_data.append((img_path, label))

        # Shuffle and split
        random.shuffle(self.all_data)
        n = len(self.all_data)
        train_end = int(0.7 * n)
        val_end = int(0.85 * n)
        if self.split == 'train':
            self.data = self.all_data[:train_end]
        elif self.split == 'val':
            self.data = self.all_data[train_end:val_end]
        elif self.split == 'test':
            self.data = self.all_data[val_end:]

    def _get_garbage_classification_label(self, category):
        if category in ['cardboard', 'glass', 'metal', 'paper', 'plastic']:
            return 1  # recyclable
        elif category == 'trash':
            return 2 if self.num_classes == 3 else None
        return None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


In [15]:
# let's test this code

# test the dataset and dataloader
transform = transforms.Compose([
    transforms.Resize(IMAGE_OUTPUT_SIZE),
    transforms.ToTensor(),
])

root_dir = 'datasets'

for num_classes in [2, 3]:
    for split in ['train', 'val', 'test']:
        dataset = CombinedWasteDataset(root_dir, num_classes=num_classes,
                                        split=split, transform=transform)
        dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
        print(f"Num classes: {num_classes}, Split: {split}, Samples: {len(dataset)}")
        if len(dataset) > 0:
            # Test one batch
            for images, labels in dataloader:
                print(f"  Batch shape: {images.shape}, Labels: {labels}")
                break

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/garbage_classification'

In [ ]:
# We can also visually make sure that everything is ok, and data looks "proper"

import numpy as np
# import torch
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader


# Create the dataset for visualization
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset_viz = CombinedWasteDataset(
    root_dir='datasets', num_classes=2, split='train', transform=transform
)

In [ ]:
# Get some random samples
num_samples = 15
random_indices = np.random.choice(len(dataset_viz), num_samples, replace=False)

fig, axes = plt.subplots(3, 5, figsize=(15, 10))

for i, idx in enumerate(random_indices):
    image, label = dataset_viz[idx]
    # Convert tensor to numpy for plotting
    image_np = image.permute(1, 2, 0).numpy()
    row = i // 5
    col = i % 5
    axes[row, col].imshow(image_np)
    axes[row, col].set_title(f'Label: {label}')
    axes[row, col].axis('off')

plt.show()

In [ ]:
# Great, everything works, we can now start creating code for training.
# First some boilerplate and configs

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms




In [ ]:
# %%
# set up the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
# %% set up the dataset and dataloaders

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = CombinedWasteDataset(
    root_dir='datasets', num_classes=2, split='train', transform=transform
)
val_dataset = CombinedWasteDataset(
    root_dir='datasets', num_classes=2, split='val', transform=transform
)

test_dataset = CombinedWasteDataset(
    root_dir='datasets', num_classes=2, split='test', transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
for images, labels in train_loader:
    print(f'Batch shape: {images.shape}')
    break

# why is the batch shape like this?
# batch size, channels, height, width
# is having channels first important? what if it was channels last?
# In PyTorch, the default tensor format is channels first (NCHW), which is optimized for performance on GPUs.
# If we had channels last (NHWC), we would need to transpose the tensor before passing it to the model, which could introduce additional overhead.


In [ ]:
# %%
# We can now try to define a simple MLP network
model = nn.Sequential(
    nn.Flatten(),  # first step - flatten the 3D image to 1D vector
    nn.Linear(3 * 224 * 224, 512),  # we have to know the input size, and decide the output size
    nn.ReLU(),
    nn.Linear(512, 256),  # again, decide the output size
    nn.ReLU(),
    nn.Linear(256, 2)   # final layer - output size must match the number of classes
    # no activation here, why?

)

In [ ]:
#%%
# lets see a summary of the model
# we can just "print" the model
print(model)


In [ ]:
# or we can do something more advanced
from torchsummary import summary
summary(model, (3, 224, 224), device="cpu")

# questions:
# - how many parameters does the model have?
# - which layers have the most parameters?
# - what are the "trainable parameters"?
#  = "Trainable parameters" are the parameters (weights and biases) in the model that are updated during training.
#  = These are typically the parameters in the layers with learnable weights, such as Linear layers.
#  = In contrast, parameters in layers like ReLU are not trainable, as they do not have weights to update.


In [ ]:
# one more way to show the model architecture
# we would need to install torchviz AND graphviz on the system
# only if we REALLY need to see our model graphically

from torchviz import make_dot
sample_input = torch.randn(1, 3, 224, 224)
sample_output = model(sample_input)

# view directly in jupyter notebook
make_dot(sample_output, params=dict(model.named_parameters()))
# or save to file
# make_dot(sample_output, params=dict(model.named_parameters())).render("model_architecture", format="png")


In [ ]:
# let's move the whole model architecture to the proper device
model.to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss() # why this loss function? and does it need an activation function at the end of our architecture?
# could we also use torch.nn.BCEWithLogitsLoss()?
# loss_fn = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(model.parameters()) # what could the "optimizer" be?; Adam, SGD, RMSprop, ... what are those?


# lets define the simplest possible training and validation loops
# what elements do we need?
# - we need the model
# - we need the data (train_loader, val_loader)
# - we need the loss function
# - we need the optimizer
# - we need to set the model to training or evaluation mode
# - we need to move data to the device (cpu or gpu)
# - we need to zero the gradients
#    = by default, gradients are accumulated in PyTorch, so we need to zero them before each backward pass
#    = this is different from TensorFlow/Keras (gradients are computed from scratch each time)
#    = but what does "accumulated" mean in this context?
#    = it means that gradients are summed over multiple backward passes, and it is not usually what we want
# - we need to do the forward pass
# - we need to compute the loss
# - we need to do the backward pass
# - we need to update the weights
# - we need to track the loss and accuracy

In [ ]:
# define the training function
def train_one_epoch(model, train_loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(images)

        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


In [ ]:
# and now the validation function
def validate(model, val_loader, loss_fn, device):
    model.eval() # what does "setting model to the evaluation mode" do?
    total_loss = 0
    correct = 0
    total = 0

# with torch.no_grad(): can also be used, but it has other implications - see the docs
    with torch.inference_mode():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total
    return total_loss / len(val_loader), accuracy

In [ ]:
# now we can run the loops,
# let's decide for how many epochs
# = what is an epoch?
# = an epoch is one complete pass through the entire training dataset (simple for now, but more complex in practice, e.g., with data augmentation {explained later} )
num_epochs = 5
for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        model, train_loader, loss_fn, optimizer, device
    )
    val_loss, val_acc = validate(model, val_loader, loss_fn, device)
    print(
        f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, '
        f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}'
    )


In [ ]:
# Test the model
test_loss, test_acc = validate(model, test_loader, loss_fn, device)
print(f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


In [ ]:
# generate a confusion matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
all_labels = []
all_preds = []
with torch.inference_mode():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Class 0 - organic', 'Class 1 - recyclable'], yticklabels=['Class 0 - organic', 'Class 1 - recyclable'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


## Doing better

The model is training, which is a good first step. Let's start making our code better.
We can rewrite the model architecture, to be more "proper", by creating a class and not keeping it as a collection of loose layers

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 224 * 224, 512)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(512, 256)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(256, 2)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x


In [ ]:
# OR
from collections import OrderedDict

class SimpleMLP_named(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(OrderedDict([
            ('flatten', nn.Flatten()),
            ('input', nn.Linear(3 * 224 * 224, 512)),
            ('relu1', nn.ReLU()),
            ('hidden', nn.Linear(512, 256)),
            ('relu2', nn.ReLU()),
            ('output', nn.Linear(256, 2))
        ]))

    def forward(self, x):
        return self.layers(x)

In [ ]:
# We can now try to define a simple MLP network
model = SimpleMLP()
model_named = SimpleMLP_named()

In [ ]:
# lets see a summary of the model
# we can just "print" the model
print(model)
print(model_named)



In [ ]:
# or we can do something more advanced
# take note - torch summary ignores our custom names
from torchsummary import summary
summary(model, (3, 224, 224), device="cpu")
summary(model_named, (3, 224, 224), device="cpu")

In [ ]:
# training would be the same, just reorganizing code

## Changing the architecture

Though we could probably have "ok" results with an MLP, we can do something more advanced - by using convolutions


In [ ]:
# creating a simple CNN network
# question - how does it work, and how to calculate the output size of each layer?
# in pure pytorch it is necessary to do this by hand (in contrast to keras, where is is done automatically)
# there are layers in pytorch that can do this for you, but they are not commonly used
# so it is a good exercise to do it by hand (especially as the more advanced architectures demand this knowledge)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(64 * 56 * 56, 512),
            nn.ReLU(),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
model = SimpleCNN()

In [ ]:
from torchsummary import summary
summary(model, (3, 224, 224), device="cpu")

In [ ]:
model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# Let's train the new model. Boilerplate code can stay the same
from tqdm import tqdm

num_epochs = 10

# adding tqdm to the loop, at least some visualization of the progress
for epoch in tqdm(range(num_epochs)):
    train_loss = train_one_epoch(
        model, train_loader, loss_fn, optimizer, device
    )
    val_loss, val_acc = validate(model, val_loader, loss_fn, device)
    print(
        f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, '
        f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}'
    )


## Printing critical information during training
We now have a prototype of an architecture, but training process is not verbose enough. We do not know what is going on, and how we are doing without waiting for the whole Epoch to end. Moreover, do we have to wait for the whole training to finish, what if the model "overfits" or we could just stop it earlier?

Secondly, we cannot even save the model and we must fix that as soon as possible.

In [ ]:
import time
from tqdm import tqdm

In [ ]:
# Setup for checkpoints and early stopping
checkpoint_dir = 'checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
best_val_loss = float('inf')
best_val_acc = 0.0

num_epochs = 50
patience = 2
current_patience = patience


In [ ]:
# let's rework training

def train_one_epoch(model, train_loader, loss_fn, optimizer, device):
    """Train for one epoch with per-batch tqdm progress and speed reporting.

    Returns:
        avg_loss (float): average loss over batches
        epoch_time (float): seconds spent in this epoch
        iters_per_sec (float): batches processed per second
        samples_per_sec (float): samples processed per second
    """
    model.train()
    total_loss = 0.0
    processed_batches = 0
    processed_samples = 0
    start_time = time.time()

    pbar = tqdm(train_loader, total=len(train_loader), desc='Train', unit='batch', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        total_loss += batch_loss
        processed_batches += 1
        processed_samples += labels.size(0)

        elapsed = time.time() - start_time
        iters_per_sec = processed_batches / elapsed if elapsed > 0 else 0.0
        samples_per_sec = processed_samples / elapsed if elapsed > 0 else 0.0

        pbar.set_postfix({
            'batch_loss': f'{batch_loss:.4f}',
            'avg_loss': f'{(total_loss/processed_batches):.4f}',
            'iter/s': f'{iters_per_sec:.2f}',
            'samples/s': f'{samples_per_sec:.1f}'
        })

    epoch_time = time.time() - start_time
    avg_loss = total_loss / len(train_loader) if len(train_loader) > 0 else 0.0
    return avg_loss, epoch_time, iters_per_sec, samples_per_sec

In [ ]:
# as well as validation

def validate(model, val_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.inference_mode():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total
    return total_loss / len(val_loader), accuracy

In [ ]:
# the whole training loop is also different

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    train_loss, epoch_time, iters_per_sec, samples_per_sec = train_one_epoch(
        model, train_loader, loss_fn, optimizer, device
    )
    val_loss, val_acc = validate(model, val_loader, loss_fn, device)
    print(
        f'Epoch {epoch+1}/{num_epochs} - '
        f'Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, '
        f'Time: {epoch_time:.1f}s, it/s: {iters_per_sec:.2f}, samples/s: {samples_per_sec:.1f}'
    )

    # Save checkpoint after each epoch
    # raw python does not have a built-in checkpointing mechanism like some other frameworks
    # so we manually save the model state dict
    torch.save(model.state_dict(), os.path.join(checkpoint_dir, f'epoch_{epoch+1}.pth'))

    # Early stopping based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), os.path.join(checkpoint_dir, 'best_model.pth'))
        current_patience = patience
    else:
        current_patience -= 1

    # # Early stopping on validation accuracy
    # if val_acc > best_val_acc:
    #     best_val_acc = val_acc
    #     torch.save(model.state_dict(), os.path.join(checkpoint_dir, 'best_model_acc.pth'))
    #     current_patience = patience

    if current_patience == 0:
        print("Early stopping triggered.")
        break


In [ ]:
# let's test the model (on test dataset)
test_loss, test_acc = validate(model, test_loader, loss_fn, device)
print(f'Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')

In [ ]:
# make sure, that we were able to save our model
# we are loading the "weights" of the model, we need to have architecture configured in the code

checkpoint_dir = 'checkpoints'
# test loading the best model
print("Loading best model saved (based on some parameter we were tracking)")
best_model_path = os.path.join(checkpoint_dir, 'best_model.pth')
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path))
    test_loss, test_acc = validate(model, test_loader, loss_fn, device)
    print(f'Best Model - Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}')


